# Analytically solving the 1D diffusion equation with SymPy (and comparing with SciPy)

This notebook is designed for teaching materials science students.

We will study the **1D diffusion equation**

$$
\frac{\partial c}{\partial t} = D \frac{\partial^2 c}{\partial x^2},
$$

where

- $c(x,t)$ is concentration,
- $D$ is the diffusion coefficient.

We will cover:

1. The PDE and its physical meaning
2. Separation of variables
3. Analytical solution for a sinusoidal initial condition
4. Symbolic verification with SymPy
5. The Gaussian fundamental solution
6. A numerical comparison using SciPy

## 1. Imports and setup

In [ ]:
import sympy as sp
sp.init_printing()

import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

## 2. The 1D diffusion equation

The governing equation is

$$
\frac{\partial c}{\partial t} = D \frac{\partial^2 c}{\partial x^2}.
$$

This equation appears throughout materials science, for example in:

- atomic diffusion in alloys,
- heat conduction (same mathematics, different physical meaning),
- homogenization of composition fields.

We first solve it analytically using **separation of variables**.

In [ ]:
x, t, D = sp.symbols('x t D', positive=True, real=True)
c = sp.Function('c')

diffusion_eq = sp.Eq(sp.diff(c(x,t), t), D*sp.diff(c(x,t), x, 2))
diffusion_eq

## 3. Separation of variables

Assume the solution can be written as

$$
c(x,t) = X(x)T(t).
$$

Substitute into the PDE:

$$
X(x) \frac{dT}{dt} = D T(t) \frac{d^2 X}{dx^2}.
$$

Divide both sides by \(X(x)T(t)\):

$$
\frac{1}{T} \frac{dT}{dt} = D \frac{1}{X} \frac{d^2X}{dx^2}.
$$

The left-hand side depends only on \(t\), and the right-hand side depends only on \(x\), so both must equal a constant.
We choose

$$
\frac{1}{T}\frac{dT}{dt} = -\lambda, \qquad
D\frac{1}{X}\frac{d^2X}{dx^2} = -\lambda.
$$

This gives two ODEs:

$$
\frac{dT}{dt} + \lambda T = 0
$$

and

$$
\frac{d^2X}{dx^2} + \frac{\lambda}{D}X = 0.
$$

In [ ]:
lam = sp.symbols('lambda', positive=True, real=True)
X = sp.Function('X')
T = sp.Function('T')

time_eq = sp.Eq(sp.diff(T(t), t) + lam*T(t), 0)
space_eq = sp.Eq(sp.diff(X(x), x, 2) + (lam/D)*X(x), 0)

time_eq, space_eq

In [ ]:
lam = sp.symbols('lambda', positive=True, real=True)
X = sp.Function('X')
T = sp.Function('T')

# product ansatz
ansatz = X(x) * T(t)

# substitute into PDE
pde_sub = sp.Eq(
    diffusion_eq.lhs.subs(c(x, t), ansatz).doit(),
    diffusion_eq.rhs.subs(c(x, t), ansatz).doit()
)

# separated equation
sep_eq = sp.Eq(
    sp.simplify(pde_sub.lhs / ansatz),
    sp.simplify(pde_sub.rhs / ansatz)
)

# introduce separation constant
time_eq = sp.Eq(sep_eq.lhs, -lam)
space_eq = sp.Eq(sep_eq.rhs, -lam)

# standard ODE forms
time_ode = sp.Eq(sp.expand(time_eq.lhs * T(t) - time_eq.rhs * T(t)), 0)
space_ode = sp.Eq(sp.expand(space_eq.lhs * X(x) - space_eq.rhs * X(x)), 0)

time_ode = sp.simplify(time_ode)
space_ode = sp.simplify(space_ode)

time_ode, space_ode

## 4. Solve the time ODE with SymPy

In [ ]:
time_sol = sp.dsolve(time_eq)
time_sol

So the time dependence is

$$
T(t) = C e^{-\lambda t}.
$$

This tells us that each spatial mode decays exponentially in time.

## 5. Solve the spatial ODE with SymPy

In [ ]:
space_sol = sp.dsolve(space_eq)
space_sol

The spatial solution is oscillatory, which is expected for bounded-domain eigenfunctions.

For a domain $0 \le x \le L$ with boundary conditions

$$
c(0,t)=0, \qquad c(L,t)=0,
$$

the allowed eigenfunctions are

$$
X_n(x)=\sin\left(\frac{n\pi x}{L}\right),
$$

with eigenvalues

$$
\lambda_n = D\left(\frac{n\pi}{L}\right)^2.
$$

Therefore the separated solution becomes

$$
c_n(x,t)=
\sin\left(\frac{n\pi x}{L}\right)
\exp\left[-D\left(\frac{n\pi}{L}\right)^2 t\right].
$$

In [ ]:
n, L = sp.symbols('n L', positive=True, integer=True)

c_n = sp.sin(n*sp.pi*x/L) * sp.exp(-D*(n*sp.pi/L)**2 * t)
c_n

## 6. Verify the analytical solution symbolically

Now we check directly that

$$
c_n(x,t)=
\sin\left(\frac{n\pi x}{L}\right)
\exp\left[-D\left(\frac{n\pi}{L}\right)^2 t\right]
$$

satisfies the diffusion equation.

In [ ]:
check_cn = sp.simplify(
    sp.diff(c_n, t) - D*sp.diff(c_n, x, 2)
)
check_cn

If the result is `0`, the symbolic expression is an exact solution of the PDE.

## 7. Example: first Fourier mode

A particularly simple exact solution is the first mode \(n=1\):

$$
c(x,t)=
\sin\left(\frac{\pi x}{L}\right)
\exp\left[-D\left(\frac{\pi}{L}\right)^2 t\right].
$$

This is a perfect teaching example because:

- the shape stays sinusoidal,
- only the amplitude decays,
- the decay rate is set by $D$ and the wavelength.

In [ ]:
c1 = sp.sin(sp.pi*x/L) * sp.exp(-D*(sp.pi/L)**2 * t)
c1

## 8. Plot the analytical solution for several times

In [ ]:
D_val = 1.0
L_val = 1.0

xv = np.linspace(0, L_val, 300)
times = [0.0, 0.02, 0.05, 0.1, 0.2]

for tt in times:
    cv = np.sin(np.pi*xv/L_val) * np.exp(-D_val*(np.pi/L_val)**2 * tt)
    plt.plot(xv, cv, label=f"t={tt}")

plt.xlabel("x")
plt.ylabel("c(x,t)")
plt.title("Analytical solution: first diffusion mode")
plt.legend()
plt.show()

## 9. Gaussian fundamental solution

Another classic exact solution on an infinite domain is the Gaussian solution:

$$
c(x,t) = \frac{1}{\sqrt{4\pi D t}}
\exp\left(-\frac{x^2}{4Dt}\right),
\qquad t>0.
$$

This represents diffusion from an initially localized spike.

In [ ]:
c_gauss = (1/sp.sqrt(4*sp.pi*D*t)) * sp.exp(-x**2/(4*D*t))
c_gauss

Let's verify symbolically that this Gaussian also satisfies the diffusion equation.

In [ ]:
check_gauss = sp.simplify(
    sp.diff(c_gauss, t) - D*sp.diff(c_gauss, x, 2)
)
check_gauss

Again, a result of `0` confirms it is an exact analytical solution.

## 10. Plot the Gaussian solution

In [ ]:
xv = np.linspace(-2.0, 2.0, 400)
times = [0.02, 0.05, 0.1, 0.2]

for tt in times:
    cv = 1/np.sqrt(4*np.pi*D_val*tt) * np.exp(-xv**2/(4*D_val*tt))
    plt.plot(xv, cv, label=f"t={tt}")

plt.xlabel("x")
plt.ylabel("c(x,t)")
plt.title("Gaussian fundamental solution of diffusion")
plt.legend()
plt.show()

## 11. Numerical comparison using SciPy

To connect the exact solution to numerical methods, we solve the first-mode problem by the **method of lines**.

We discretize space and solve the resulting ODE system in time with `scipy.integrate.solve_ivp`.

The analytical solution is

$$
c(x,t)=
\sin\left(\frac{\pi x}{L}\right)
\exp\left[-D\left(\frac{\pi}{L}\right)^2 t\right].
$$

In [ ]:
D_val = 1.0
L_val = 1.0
Nx = 80

xv = np.linspace(0, L_val, Nx)
dx = xv[1] - xv[0]

def rhs(t, y):
    dydt = np.zeros_like(y)
    dydt[1:-1] = D_val * (y[2:] - 2*y[1:-1] + y[:-2]) / dx**2
    dydt[0] = 0.0
    dydt[-1] = 0.0
    return dydt

y0 = np.sin(np.pi*xv/L_val)
sol = solve_ivp(rhs, [0, 0.1], y0, t_eval=[0.0, 0.02, 0.05, 0.1])

sol.y.shape

In [ ]:
for k, tt in enumerate(sol.t):
    plt.plot(xv, sol.y[:,k], 'o', ms=3, label=f"numerical t={tt:.2f}")
    exact = np.sin(np.pi*xv/L_val) * np.exp(-D_val*(np.pi/L_val)**2 * tt)
    plt.plot(xv, exact, '-', lw=2, label=f"exact t={tt:.2f}")

plt.xlabel("x")
plt.ylabel("c(x,t)")
plt.title("SciPy numerical solution vs analytical solution")
plt.legend(ncol=2, fontsize=8)
plt.show()

## 12. Physical interpretation

For the sinusoidal solution,

$$
c(x,t)=
\sin\left(\frac{n\pi x}{L}\right)
\exp\left[-D\left(\frac{n\pi}{L}\right)^2 t\right],
$$

we see that:

- the **shape** is an eigenmode,
- the **amplitude decays exponentially**,
- higher-frequency modes decay faster because of the \(n^2\) factor.

This explains why diffusion smooths out sharp concentration gradients quickly.

## 13. Summary

In this notebook, we:

- wrote the 1D diffusion equation,
- solved it by separation of variables,
- used SymPy to solve the ODEs,
- verified exact analytical solutions symbolically,
- studied both a bounded-domain sine mode and the Gaussian fundamental solution,
- compared the exact result with a SciPy numerical solution.

These are core ideas for diffusion, heat conduction, and many transport problems in materials science.

## 14. Optional exercises

1. Verify the solution for the second mode \(n=2\).
2. Build a Fourier-series solution for a non-sinusoidal initial condition.
3. Change \(D\) and observe how the decay rate changes.
4. Replace concentration with temperature and reinterpret the PDE as the heat equation.